This notebook will show you how to realise a benchmark of UW Current models by comparison between modeled data and In Situ data from the <a href="https://help.marine.copernicus.eu/en/collections/4060068-copernicus-marine-toolbox">Copernicus Marine Toolbox</a>.

Commented version by JG

# Import libraries & functions

In [ ]:
%pip install copernicusmarine # Install copernicus marine library if necessary
%pip install cartopy

In [ ]:
import os

import copernicusmarine

#!pip unsintall matplotlib # uinstall if it's necessary because of version issue
#!pip install matplotlib # install if it's necessary
#!pip install cartopy # install if it's necessary
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import xarray as xr
from scipy.stats import pearsonr


# 1. Data collection and preprocessing

## 1.1. Copernicus data

 Extract in situ data from Copernicus Data Base

<div class="alert alert-block alert-warning">
Don't forget to create an account to Copernicus Marine in order to have access to In Situ Data.
</div>

### Parameters

Coordinates of the zone where we want to extract In Situ Data in order to compare the UW current

In [ ]:
# Far from Brest
minimumLongitude = -6.25
maximumLongitude = -6
minimumLatitude = 46.6
maximumLatitude = 47

minimumDepth=0.49402499198913574
maximumDepth=1000
zone_name = 'brest'

<div class="alert alert-block alert-info">
<b>Note:</b> For an easy use, you can get the value of the coordinates directly by drawing the desired zone in  
<a href="https://data.marine.copernicus.eu/product/GLOBAL_ANALYSISFORECAST_PHY_001_024/download?">Copernicus interface</a>.<br>
    You can look in the “Automate” tab then “Python API” tab to retrieve the value of the coordinates, the start time and the end time.
</div>

<div class="alert alert-block alert-info">
<b>Note:</b> In situ data can be collected point-in-time or across the wider region.
</div>

Time at which we want to extract In Situ Data in order to compare the UW current

In [ ]:
from datetime import datetime

# Créer un objet datetime
start_date_object = datetime(2023, 11, 2, 0, 0, 0) 
end_date_object = datetime(2023, 11, 4, 0, 0, 0)

# Reproduire le format 'YYYY-MM-DDTHH:MM:SS'
start_dateTime = start_date_object.strftime("%Y-%m-%dT%H:%M:%S")
end_dateTime = end_date_object.strftime("%Y-%m-%dT%H:%M:%S")

# Calcul de la durée en heures des données sauvegarder
duration_hours = int((end_date_object - start_date_object).total_seconds() / 3600)

print(start_dateTime)
print(end_dateTime)
print(duration_hours)

<div class="alert alert-block alert-info">
<b>Note:</b> In situ data can be collected at a given time or over a defined period of time.
</div>

Choose SAVE path and names of input files

In [ ]:
date_data_folder = f"{start_date_object.strftime('%Y-%m-%d')}_{duration_hours}hours/"

# Specify the path where you want to save the Copernicus Data
save_path_copernicus = os.path.join('./DataOcean', date_data_folder, zone_name, 'copernicus/')

# Specify the names of input files
nc_copernicus_wind_file = 'wind.nc'
nc_copernicus_uwcurrent_file = 'uwCurrent.nc'
nc_copernicus_bathy_file = 'bathy.nc'

# Check if path exists
if not os.path.exists(save_path_copernicus):
    # Create save path if it doesn't exist
    os.makedirs(save_path_copernicus)
    print(f"Save path '{save_path_copernicus}' has been created.")
else:
    print(f"Save path '{save_path_copernicus}' already exists.")
    

### Wind collection

The products are produced by the Royal Netherlands Meteorological
Institute (KNMI) based on the Level-3 (L3) NRT and MY wind observations from scatterometer
and on numerical weather prediction (NWP) model fields. 
https://documentation.marine.copernicus.eu/PUM/CMEMS-WIND-PUM-012-004-006.pdf

In [ ]:
copernicusmarine.subset(
  dataset_id="cmems_obs-wind_glo_phy_nrt_l4_0.125deg_PT1H",
  minimum_longitude=minimumLongitude,
  maximum_longitude=maximumLongitude,
  minimum_latitude=minimumLatitude,
  maximum_latitude=maximumLatitude,
  start_datetime=start_dateTime,
  end_datetime=end_dateTime,
  output_filename = save_path_copernicus+nc_copernicus_wind_file,
    variables = ["air_density","northward_wind","eastward_wind"]
)

### UW Current collection

"Every day the forecast delivered the previous day is overwritten by one day of
simulation and the new 10-day forecast. Weekly, the simulations are replaced by the
NRT-analyses (from -192 to -24 hr) and the NRT-analysis by the best analyses (from -
360 to -192 hr). The last 11-15 days of the historical time series of
simulation/analysis/forecast are overwritten every day."
https://documentation.marine.copernicus.eu/PUM/CMEMS-GLO-PUM-001-024.pdf

In [ ]:
copernicusmarine.subset(
    dataset_id="cmems_mod_glo_phy-cur_anfc_0.083deg_PT6H-i",
    minimum_longitude=minimumLongitude,
    maximum_longitude=maximumLongitude,
    minimum_latitude=minimumLatitude,
    maximum_latitude=maximumLatitude,
    start_datetime=start_dateTime,
    end_datetime=end_dateTime,
    minimum_depth=minimumDepth,
    maximum_depth=maximumDepth,
    output_filename = save_path_copernicus+nc_copernicus_uwcurrent_file,
    variables=["uo", "vo"],
)

### Bathymetry collection

In [ ]:
copernicusmarine.subset(
  dataset_id="cmems_mod_glo_phy_anfc_0.083deg_static",
  #force_dataset_part="bathy",
  variables=["deptho"],
  minimum_longitude=minimumLongitude,
  maximum_longitude=maximumLongitude,
  minimum_latitude=minimumLatitude,
  maximum_latitude=maximumLatitude,
  minimum_depth=0.5,
  maximum_depth=5727,
  output_filename = save_path_copernicus+nc_copernicus_bathy_file,
)

### Data preprocessing

Reading of the different files

In [ ]:
uwCurrent_file = xr.open_dataset(rf"{save_path_copernicus + nc_copernicus_uwcurrent_file}")
wind_file = xr.open_dataset(rf"{save_path_copernicus + nc_copernicus_wind_file}")
bathy_file = xr.open_dataset(rf"{save_path_copernicus + nc_copernicus_bathy_file}")

Get the raw data from nc files

In [ ]:
copernicus_uwCurrent_data = pd.DataFrame(uwCurrent_file.to_dataframe())
copernicus_wind_data = pd.DataFrame(wind_file.to_dataframe())
copernicus_bathy_data = pd.DataFrame(bathy_file.to_dataframe())

Delete NaN from data

In [ ]:
copernicus_uwCurrent_data = copernicus_uwCurrent_data.dropna().reset_index()
copernicus_wind_data = copernicus_wind_data.dropna().reset_index()
copernicus_bathy_data = copernicus_bathy_data.dropna().reset_index()

In [ ]:
copernicus_uwCurrent_data

Sort data by latitude longitude

In [ ]:
copernicus_uwCurrent_data = copernicus_uwCurrent_data.sort_values(by=['time', 'latitude', 'longitude'])
copernicus_wind_data = copernicus_wind_data.sort_values(by=['time', 'latitude', 'longitude'])
copernicus_bathy_data = copernicus_bathy_data.sort_values(by=['latitude', 'longitude'])

<div class="alert alert-block alert-info">
<b>Note:</b> You can prepare and clean the data like previously just by this command line: <br>
<i>uwCurrent_data = uwCurrent_data.dropna().reset_index().sort_values(by=['latitude', 'longitude'])</i>
</div>

Convert data latlong coordinates into cartesian coordinates

In [ ]:
#!pip install pyproj # install this library if it's necessary
import pyproj


def lat_lon_to_cartesian(lat, lon):
    """
    Convert latitude and longitude to Cartesian coordinates using pyproj.
    
    Parameters:
    lat (float): Latitude in degrees.
    lon (float): Longitude in degrees.
    
    Returns:
    tuple: Cartesian coordinates (x, y, z).
    """
    # Create the WGS84 CRS (coordinate reference system for lat/lon)
    wgs84 = pyproj.CRS('EPSG:4326')  # WGS84 (latitude/longitude)
    ecef = pyproj.CRS('EPSG:4978')  # Earth-Centered, Earth-Fixed (ECEF)
    
    # Create a transformer to convert lat/lon to Cartesian (ECEF)
    transformer = pyproj.Transformer.from_crs(wgs84, ecef, always_xy=True)
    
    # Convert lat/lon to x, y, z
    result = transformer.transform(lon, lat)  # Lon, Lat order
    x, y = result  # Only unpack the first two values
    
    # If 'z' is not returned, set it to 0 by default
    z = 0  # Default z value
    
    return x, y, z

def convert_df_to_cartesian(df):
    """
    Convert a DataFrame with latitude and longitude columns to Cartesian coordinates.
    
    Parameters:
    df (DataFrame): DataFrame with 'latitude' and 'longitude' columns.
    
    Returns:
    DataFrame: Original DataFrame with additional 'x', 'y', 'z' columns.
    """
    # Apply the lat_lon_to_cartesian function to each row
    df[['x', 'y', 'z']] = df.apply(lambda row: pd.Series(lat_lon_to_cartesian(row['latitude'], row['longitude'])), axis=1)
    return df


In [ ]:
# Add cartesian coordinates to the dataframes
copernicus_uwCurrent_data_cart = convert_df_to_cartesian(copernicus_uwCurrent_data)
copernicus_wind_data_cart = convert_df_to_cartesian(copernicus_wind_data )
copernicus_bathy_data_cart = convert_df_to_cartesian(copernicus_bathy_data )

# Print the columns of dataframes with cartesian coordinates
print(copernicus_uwCurrent_data_cart.columns)
print(copernicus_wind_data_cart.columns)
print(copernicus_bathy_data_cart.columns)

### Saving in CSV format

Save LatLong DataFrames in CSV format

In [ ]:
# Specify the names of input files
copernicus_wind_file_CSV = 'wind_latlong.csv'
copernicus_uwcurrent_file_CSV = 'uwCurrent_latlong.csv'
copernicus_bathy_file_CSV = 'bathy_latlong.csv'

# Save dataframes in CSV format
copernicus_uwCurrent_data.to_csv(rf"{save_path_copernicus + copernicus_uwcurrent_file_CSV}")
copernicus_wind_data.to_csv(rf"{save_path_copernicus + copernicus_wind_file_CSV}")
copernicus_bathy_data.to_csv(rf"{save_path_copernicus + copernicus_bathy_file_CSV}")

Save cartesian DataFrames in CSV format

In [ ]:
# Specify the names of input files
copernicus_wind_cart_file_CSV = 'wind_cart.csv'
copernicus_uwcurrent_cart_file_CSV = 'uwCurrent_cart.csv'
copernicus_bathy_cart_file_CSV = 'bathy_cart.csv'

# Save dataframes in CSV format
copernicus_uwCurrent_data_cart.to_csv(rf"{save_path_copernicus + copernicus_uwcurrent_cart_file_CSV}")
copernicus_wind_data_cart.to_csv(rf"{save_path_copernicus + copernicus_wind_cart_file_CSV}")
copernicus_bathy_data_cart.to_csv(rf"{save_path_copernicus + copernicus_bathy_cart_file_CSV}")

Save bathy with cartesian coordinates in heightmap.png format

In [ ]:

# Create image Heightmap PNG
def create_heightmap(X, Y, Z, nX, nY, interpolationMethod, image_sortie):
    # Define regular grid size
    min_x, max_x = np.min(X), np.max(X)
    min_y, max_y = np.min(Y), np.max(Y)
    
    # Create 2D grid for X,Y coordinates
    x_grid = np.linspace(min_x, max_x, nX)
    y_grid = np.linspace(min_y, max_y, nY)
    X_grid, Y_grid = np.meshgrid(x_grid, y_grid)
    
    # Interpolation of Z values on grid (use of 'linear' method here)
    from scipy.interpolate import griddata
    Z_grid = griddata((X, Y), Z, (X_grid, Y_grid), method=interpolationMethod)
    
    # Create image (use colormap to represent depth)
    plt.imshow(Z_grid, extent=(min_x, max_x, min_y, max_y), origin='lower', cmap='gray')
    plt.colorbar(label='Depth')
    plt.title('Heightmap Depth')
    plt.savefig(image_sortie, format='png', dpi=300)
    print(f"Image Heightmap enregistrée sous {image_sortie}")


nX = 500
nY = 500
interpolationMethod = 'linear'

# Name of the PNG image
image_bathy_file= 'bathy_cartesian.png'

create_heightmap(copernicus_bathy_data_cart['x'], copernicus_bathy_data_cart['y'], copernicus_bathy_data_cart['deptho'], nX, nY, interpolationMethod, rf"{save_path_copernicus + image_bathy_file}")

### Loading output data directly

If you already have the data, use this cell:

In [ ]:
# Copernicus files
copernicus_wind_file_CSV = 'wind_latlong.csv'
copernicus_uwcurrent_file_CSV = 'uwCurrent_latlong.csv'
copernicus_bathy_file_CSV = 'bathy_latlong.csv'

# Load files
copernicus_uwCurrent_data = pd.read_csv(rf"{save_path_copernicus + copernicus_uwcurrent_file_CSV}")
copernicus_wind_data = pd.read_csv(rf"{save_path_copernicus + copernicus_wind_file_CSV}")
copernicus_bathy_data = pd.read_csv(rf"{save_path_copernicus + copernicus_bathy_file_CSV}")

In [ ]:
# Copernicus files
copernicus_wind_cart_file_CSV = 'wind_cart.csv'
copernicus_uwcurrent_cart_file_CSV = 'uwCurrent_cart.csv'
copernicus_bathy_cart_file_CSV = 'bathy_cart.csv'

# Load files
copernicus_uwCurrent_data_cart = pd.read_csv(rf"{save_path_copernicus + copernicus_uwcurrent_cart_file_CSV}")
copernicus_wind_data_cart = pd.read_csv(rf"{save_path_copernicus + copernicus_wind_cart_file_CSV}")
copernicus_bathy_data_cart = pd.read_csv(rf"{save_path_copernicus + copernicus_bathy_cart_file_CSV}")

### Output data copernicus 

In [ ]:
copernicus_uwCurrent_data_cart_filtered = copernicus_uwCurrent_data_cart.copy()

copernicus_uwCurrent_data_cart_filtered ["date"] = pd.to_datetime(copernicus_uwCurrent_data_cart_filtered ["time"])

# Normalize 'date' column to remove time (hour, minute, second)
copernicus_uwCurrent_data_cart_filtered['date'] = copernicus_uwCurrent_data_cart_filtered['date'].dt.date

# Convert the "time" column to only show the time (without the date)
copernicus_uwCurrent_data_cart_filtered['time'] = pd.to_datetime(copernicus_uwCurrent_data_cart_filtered['time']).dt.time

# Now normalize start_dateTime_dt to remove hour/minute/second
start_dateTime_dt = pd.to_datetime(start_dateTime).normalize().date()

# Filter the DataFrame based on the date
copernicus_output_table = copernicus_uwCurrent_data_cart_filtered[
    copernicus_uwCurrent_data_cart_filtered["date"] != start_dateTime_dt
]

# Find the latest date in the DataFrame
latest_date = copernicus_output_table["date"].max()

# Filter out rows with the latest date
copernicus_output_table = copernicus_output_table[
    copernicus_output_table["date"] != latest_date
]

# Assuming df is your DataFrame
copernicus_output_table = copernicus_output_table.drop(columns=['Unnamed: 0'])  # Drop the first and second columns

# Reorder columns
copernicus_uwCurrent_data_final = copernicus_output_table[['depth', 'latitude', 'longitude', 'date', 'time', 'uo', 'vo', 'x', 'y', 'z']]

print("Copernicus output table for Gauss-Markov process:")
print(copernicus_uwCurrent_data_final)

## 1.2. Data from Gauss-Markov (Plugin UUV Simulator)

### Objective: Running the Gauss-Markov Process for UUVSim Using Copernicus Data

The plugin **UnderwaterCurrentPlugin** integrates the Gauss-Markov process into the simulation to model underwater currents based on predefined parameters, with data output to a CSV file at specified simulation times. The use of Copernicus ocean current data as input helps in providing realistic ocean current simulations for the underwater vehicle model.

To maintain consistency in evaluation, in situ data (Copernicus data) were retrieved across a depth range from approximately 0.5 meters to 1000 meters. This range aligns with the typical depth layers relevant to surface and near-surface current dynamics, ensuring a consistent comparison between observational and simulated data. The observational data were collected at four time intervals—00h, 06h, 12h, and 18h—on 02/06/2024, with the corresponding data from the following day (03/06/2024) used as a baseline to evaluate the Gauss-Markov predictions from the UUV simulator and the results from our Ekman model (see 1.3 Ekman model).

### Parameters

In [ ]:
date_data_folder = f"{start_date_object.strftime('%Y-%m-%d')}_{duration_hours}hours/"

# Specify the path where you want to save the Copernicus Data
save_path_gaussMarkov = os.path.join('./DataOcean', date_data_folder, zone_name, 'gaussMarkov/') 

# Check if path exists
if not os.path.exists(save_path_gaussMarkov):
    # Create save path if it doesn't exist
    os.makedirs(save_path_gaussMarkov)
    print(f"Save path '{save_path_gaussMarkov}' has been created.")
else:
    print(f"Save path '{save_path_gaussMarkov}' already exists.")

### Step 1: Building the data

In [ ]:
# Copy the dataframe
copernicus_uwCurrent_data_cart_filtered = copernicus_uwCurrent_data_cart.copy()

copernicus_uwCurrent_data_cart_filtered ["date"] = pd.to_datetime(copernicus_uwCurrent_data_cart_filtered ["time"])
# Normalize the 'date' column to remove time (similar to what you did for start_dateTime)
copernicus_uwCurrent_data_cart_filtered ['date'] = pd.to_datetime(copernicus_uwCurrent_data_cart_filtered ['date']).dt.normalize()

# Filter data from the first value of time
copernicus_uwCurrent_data_cart_filtered = copernicus_uwCurrent_data_cart_filtered[copernicus_uwCurrent_data_cart_filtered["date"] >= start_dateTime]

# Compute velocity as the magnitude of the vector (u0, v0) at each row
#copernicus_uwCurrent_data_cart_filtered["velocity"] = np.sqrt(np.square(copernicus_uwCurrent_data_cart_filtered["uo"]) + np.square(copernicus_uwCurrent_data_cart_filtered["vo"]))

print(copernicus_uwCurrent_data_cart_filtered)

In [ ]:
# Compute speed from uo (eastward velocity) and vo (northward velocity)
copernicus_uwCurrent_data_cart_filtered["velocity"] = np.sqrt(
    copernicus_uwCurrent_data_cart_filtered["uo"] ** 2 + copernicus_uwCurrent_data_cart_filtered["vo"] ** 2
)

# Compute horizontal angle (bearing in radians)
copernicus_uwCurrent_data_cart_filtered["horizontal_angle"] = np.arctan2(
    copernicus_uwCurrent_data_cart_filtered["vo"], copernicus_uwCurrent_data_cart_filtered["uo"]
)

# Estimate vertical angle (assuming small angles, we can use depth variation)
copernicus_uwCurrent_data_cart_filtered["vertical_angle"] = np.gradient(
    copernicus_uwCurrent_data_cart_filtered["depth"]
)

In [ ]:
# Compute statistics

# The std() method calculates the standard deviation, which is a measure of how much the data fluctuates around the mean. 
# In the context of your simulation parameters, noiseAmp represents the amplitude of noise, 
# which can be interpreted as the natural variability or spread of the velocity and angle data.

velocity_stats = {
    "mean": copernicus_uwCurrent_data_cart_filtered["velocity"].mean(),
    "min": copernicus_uwCurrent_data_cart_filtered["velocity"].min(),
    "max": copernicus_uwCurrent_data_cart_filtered["velocity"].max(),
    "mu": 0.0,  # Placeholder for user-defined parameter
    "noiseAmp": copernicus_uwCurrent_data_cart_filtered["velocity"].std(),
}

horizontal_angle_stats = {
    "mean": copernicus_uwCurrent_data_cart_filtered["horizontal_angle"].mean(),
    "min": copernicus_uwCurrent_data_cart_filtered["horizontal_angle"].min(),
    "max": copernicus_uwCurrent_data_cart_filtered["horizontal_angle"].max(),
    "mu": 0.0,  # Placeholder
    "noiseAmp": copernicus_uwCurrent_data_cart_filtered["horizontal_angle"].std(),
}

vertical_angle_stats = {
    "mean": copernicus_uwCurrent_data_cart_filtered["vertical_angle"].mean(),
    "min": copernicus_uwCurrent_data_cart_filtered["vertical_angle"].min(),
    "max": copernicus_uwCurrent_data_cart_filtered["vertical_angle"].max(),
    "mu": 0.0,  # Placeholder
    "noiseAmp": copernicus_uwCurrent_data_cart_filtered["vertical_angle"].std(),
}

# Print results
print("Velocity Stats:", velocity_stats)
print("Horizontal Angle Stats:", horizontal_angle_stats)
print("Vertical Angle Stats:", vertical_angle_stats)

### Step 2: Running the Gauss-Markov process

**Explanation of the plugin UnderwaterCurrentPlugin**: The idea is to extract ocean current data from the Copernicus dataset (represented as uo and vo components for East-West and North-South velocities) at four specific times (00:00, 06:00, 12:00, 18:00), calculate their mean values over an 18-hour period, and use those values as input parameters for the Gauss-Markov model.

Here’s the outline of the steps for data extraction and preparation:

- Fetch Data from Copernicus: Obtain the current values (uo, vo) for specific times (00:00, 06:00, 12:00, 18:00) of a specific date.
- Compute Mean: Calculate the mean values of the components (uo_mean, vo_mean) over the 18-hour period.
- Prepare the Input Parameters: Use the computed mean values to set the model parameters for the Gauss-Markov process in the auv_controls_2.sdf file.
- SDF Configuration: Use the extracted mean ocean current values (velocity, angles) to populate the constant_current section in the SDF file that configures the underwater current simulation plugin.

The file `auv_controls_2.sdf` will be configured with values like the velocity mean and noise amplitude derived from the Copernicus dataset.

**To simulate**:
- In the first terminal:
  - `export GZ_SIM_SYSTEM_PLUGIN_PATH=<path/to>/gauss_markov_uuvsim/build`
  - `gz sim '<path/to>/gauss_markov_uuvsim/auv_controls_2.sdf'`

- In the second terminal, to see the values of \ocean_current :
  - `gz topic -e -t \ocean_current`

#### Loading output results data directly

In [ ]:
# Gauss-Markov output file
gauss_markov_output_file_CSV = 'gauss_markov_output_data.csv'

# Copernicus input file of the simulation 
#copernicus_input_file_CSV = 'copernicus_output_data.csv'

# Load files
gauss_markov_uwCurrent_data_final = pd.read_csv(rf"{save_path_gaussMarkov + gauss_markov_output_file_CSV}")
#copernicus_uwCurrent_data_final = pd.read_csv(r"{}".format(save_path_gaussMarkov + copernicus_input_file_CSV))

#### Post-processing Gauss-Markov data

In [ ]:
# Compute uo and vo
gauss_markov_uwCurrent_data_final["uo"] = np.sqrt(gauss_markov_uwCurrent_data_final["CurrentVelocityX"]**2 + gauss_markov_uwCurrent_data_final["CurrentVelocityZ"]**2)
gauss_markov_uwCurrent_data_final["vo"] = np.sqrt(gauss_markov_uwCurrent_data_final["CurrentVelocityY"]**2 + gauss_markov_uwCurrent_data_final["CurrentVelocityZ"]**2)

print(gauss_markov_uwCurrent_data_final)

In [ ]:
gauss_markov_expanded = copernicus_uwCurrent_data_final.copy()
uo_minuit = gauss_markov_uwCurrent_data_final.loc[gauss_markov_uwCurrent_data_final['simTimeSec'] == 21600, 'uo']
vo_minuit = gauss_markov_uwCurrent_data_final.loc[gauss_markov_uwCurrent_data_final['simTimeSec'] == 21600, 'vo']

uo_1 = gauss_markov_uwCurrent_data_final.loc[gauss_markov_uwCurrent_data_final['simTimeSec'] == 43200, 'uo']
vo_1 = gauss_markov_uwCurrent_data_final.loc[gauss_markov_uwCurrent_data_final['simTimeSec'] == 43200, 'vo']

uo_2 = gauss_markov_uwCurrent_data_final.loc[gauss_markov_uwCurrent_data_final['simTimeSec'] == 64800, 'uo']
vo_2 = gauss_markov_uwCurrent_data_final.loc[gauss_markov_uwCurrent_data_final['simTimeSec'] == 64800, 'vo']

uo_3 = gauss_markov_uwCurrent_data_final.loc[gauss_markov_uwCurrent_data_final['simTimeSec'] == 86400, 'uo']
vo_3 = gauss_markov_uwCurrent_data_final.loc[gauss_markov_uwCurrent_data_final['simTimeSec'] == 86400, 'vo']

# Convert the 'time' column to string format 'HH:MM:SS'
gauss_markov_expanded['time'] = gauss_markov_expanded['time'].apply(lambda x: x.strftime('%H:%M:%S'))

# Now assign the values for uo and vo
gauss_markov_expanded.loc[gauss_markov_expanded['time'] == '00:00:00', 'uo'] = uo_minuit.iloc[0]
gauss_markov_expanded.loc[gauss_markov_expanded['time'] == '00:00:00', 'vo'] = vo_minuit.iloc[0]

gauss_markov_expanded.loc[gauss_markov_expanded['time'] == '06:00:00', 'uo'] = uo_1.iloc[0]
gauss_markov_expanded.loc[gauss_markov_expanded['time'] == '06:00:00', 'vo'] = vo_1.iloc[0]

gauss_markov_expanded.loc[gauss_markov_expanded['time'] == '12:00:00', 'uo'] = uo_2.iloc[0]
gauss_markov_expanded.loc[gauss_markov_expanded['time'] == '12:00:00', 'vo'] = vo_2.iloc[0]

gauss_markov_expanded.loc[gauss_markov_expanded['time'] == '18:00:00', 'uo'] = uo_3.iloc[0]
gauss_markov_expanded.loc[gauss_markov_expanded['time'] == '18:00:00', 'vo'] = vo_3.iloc[0]

In [ ]:
# Reindex gauss_markov_expanded to match the index of copernicus_uwCurrent_data_final
gauss_markov_expanded = gauss_markov_expanded.reindex(copernicus_uwCurrent_data_final.index)

In [ ]:
gauss_markov_expanded 

## 1.3 Data from Ekman model

#### Parameters

In [ ]:
date_data_folder = f"{start_date_object.strftime('%Y-%m-%d')}_{duration_hours}hours/"

# Specify the path where you want to save the Copernicus Data
save_path_ekman = os.path.join('./DataOcean', date_data_folder, zone_name, 'ekman/')

# Check if path exists
if not os.path.exists(save_path_ekman):
    # Create save path if it doesn't exist
    os.makedirs(save_path_ekman)
    print(f"Save path '{save_path_ekman}' has been created.")
else:
    print(f"Save path '{save_path_ekman}' already exists.")

### Building the data for the input of Ekman model

##### UW Current

In [ ]:
# Copy the dataframe
ekman_uwCurrent_input_cart = copernicus_uwCurrent_data_cart.copy()

ekman_uwCurrent_input_cart["date"] = pd.to_datetime(ekman_uwCurrent_input_cart["time"])
# Normalize the 'date' column to remove time (similar to what you did for start_dateTime)
ekman_uwCurrent_input_cart['date'] = pd.to_datetime(ekman_uwCurrent_input_cart['date']).dt.normalize()

# Filter data from the first value of time
ekman_uwCurrent_input_cart = ekman_uwCurrent_input_cart[ekman_uwCurrent_input_cart["date"] >= start_dateTime]

# Get the earliest date in the DataFrame
first_date = ekman_uwCurrent_input_cart["date"].min()

# Filter to keep only the rows with the first date
ekman_uwCurrent_input_cart = ekman_uwCurrent_input_cart[ekman_uwCurrent_input_cart["date"] == first_date]

# Assuming df is your DataFrame
ekman_uwCurrent_input_cart = ekman_uwCurrent_input_cart.drop(columns=['Unnamed: 0'])  # Drop the first and second columns

In [ ]:
ekman_uwCurrent_input_cart

##### Wind 

In [ ]:
# Copy the wind dataframe
ekman_wind_input_cart = copernicus_wind_data.copy()

ekman_wind_input_cart["date"] = pd.to_datetime(ekman_wind_input_cart["time"])
# Normalize the 'date' column to remove time (similar to what you did for start_dateTime)
ekman_wind_input_cart['date'] = pd.to_datetime(ekman_wind_input_cart['date']).dt.normalize()

In [ ]:
# Copy the wind dataframe
ekman_wind_input_cart = copernicus_wind_data.copy()

ekman_wind_input_cart["date"] = pd.to_datetime(ekman_wind_input_cart["time"])
# Normalize the 'date' column to remove time (similar to what you did for start_dateTime)
ekman_wind_input_cart['date'] = pd.to_datetime(ekman_wind_input_cart['date']).dt.normalize()

# Filter data from the first value of time
ekman_wind_input_cart = ekman_wind_input_cart[ekman_wind_input_cart["date"] >= start_dateTime]

# Filter to keep only the rows with the first date
ekman_wind_input_cart = ekman_wind_input_cart[ekman_wind_input_cart["date"] == first_date]

# Assuming df is your DataFrame
ekman_wind_input_cart = ekman_wind_input_cart.drop(columns=['Unnamed: 0'])  # Drop the first and second columns

In [ ]:
ekman_wind_input_cart

##### Compute mean uwCurrent and mean wind for the setting of the simulator python_ekman/classes/utils/Constants.py

In [ ]:
# Compute mean uwCurrent and mean wind for the setting of the simulator python_ekman/classes/utils/Constants.py
au=np.mean(ekman_uwCurrent_input_cart['uo'].values)
av=np.mean(ekman_uwCurrent_input_cart['vo'].values)
print(f"currentMeanU: {au}")
print(f"currentMeanV: {av}")
au=np.mean(ekman_wind_input_cart['eastward_wind'].values)
av=np.mean(ekman_wind_input_cart['northward_wind'].values)
print(f"v_north: {av}")
print(f"v_east: {au}")

### Loading output results data directly

In [ ]:
ekman_uwcurrent_cart_file_CSV = 'uwCurrent_output_cart.csv'
ekman_uwCurrent_output_cart = pd.read_csv(rf"{save_path_ekman+ ekman_uwcurrent_cart_file_CSV}")

# Assuming df is your DataFrame
ekman_uwCurrent_data_final = ekman_uwCurrent_output_cart.drop(columns=['Unnamed: 0', 'Unnamed: 0.1'])  # Drop the first and second columns

ekman_uwCurrent_data_final.sort_values(by='depth')

In [ ]:
ekman_uwCurrent_data_final['time'] = ekman_uwCurrent_data_final['time'].apply(lambda x: str(pd.to_timedelta(x - 21600, unit='s')))

# Keep only HH:MM:SS format (remove "0 days" part)
ekman_uwCurrent_data_final['time'] = ekman_uwCurrent_data_final['time'].str.split().str[-1]
ekman_uwCurrent_data_final

# 2. Univariate statistics

## 2.1. Summary statistics

### Copernicus

In [ ]:
copernicus_uwCurrent_data_final.loc[:,["uo","vo"]].describe().round(2).T

### Gaussian model

In [ ]:
gauss_markov_uwCurrent_data_final.loc[:,["uo","vo"]].describe().round(2).T

In [ ]:
gauss_markov_expanded.loc[:,["uo","vo"]].describe().round(2).T

### Ekman model

In [ ]:
ekman_uwCurrent_data_final.loc[:,["uo","vo"]].describe().round(2).T

## 2.2. Location 

### Copernicus

In [ ]:
copernicus_uwCurrent_data_final = copernicus_uwCurrent_data_final.sort_values(["depth","x","y"]).reset_index()
copernicus_uwCurrent_data_final_loc = copernicus_uwCurrent_data_final.loc[:,["depth","x","y"]]
copernicus_uwCurrent_data_final_loc

### Ekman

In [ ]:
ekman_uwCurrent_data_final = ekman_uwCurrent_data_final.sort_values(["depth","x","y"]).reset_index()
ekman_uwCurrent_data_final_loc = ekman_uwCurrent_data_final.loc[:,["depth","x","y"]]
ekman_uwCurrent_data_final_loc 

In [ ]:
np.sum((ekman_uwCurrent_data_final_loc == copernicus_uwCurrent_data_final_loc) == False)

## 2.2 Normality

### Densities

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 6))

# Plot the density of 'uo' on the first subplot
copernicus_uwCurrent_data_final['uo'].plot(kind='density', title=" ", ax=axes[0])
axes[0].legend()  # Adding legend to the first subplot

# Plot the density of 'vo' on the second subplot
copernicus_uwCurrent_data_final['vo'].plot(kind='density', title=" ", ax=axes[1],color='green')
axes[1].legend()  # Adding legend to the first subplot

# Adjust layout to avoid overlap
plt.tight_layout()

fig.suptitle("Density of Copernicus Data", fontsize=16)


# Show thenor plot
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 6))

# Plot the density of 'uo' on the first subplot
gauss_markov_uwCurrent_data_final['uo'].plot(kind='density', title=" ", ax=axes[0])
axes[0].legend()  # Adding legend to the first subplot

# Plot the density of 'vo' on the second subplot
gauss_markov_uwCurrent_data_final['vo'].plot(kind='density', title=" ", ax=axes[1],color='green')
axes[1].legend()  # Adding legend to the first subplot

# Adjust layout to avoid overlap
plt.tight_layout()

fig.suptitle("Density of Gaussian Markov Data", fontsize=16)
# Show the plot
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 6))

# Plot the density of 'uo' on the first subplot
gauss_markov_expanded['uo'].plot(kind='density', title=" ", ax=axes[0])
axes[0].legend()  # Adding legend to the first subplot

# Plot the density of 'vo' on the second subplot
gauss_markov_expanded['vo'].plot(kind='density', title=" ", ax=axes[1],color='green')
axes[1].legend()  # Adding legend to the first subplot

# Adjust layout to avoid overlap
plt.tight_layout()

fig.suptitle("Density of Gaussian Markov Data", fontsize=16)
# Show the plot
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 6))

# Plot the density of 'uo' on the first subplot
ekman_uwCurrent_data_final['uo'].plot(kind='density', title=" ", ax=axes[0])
axes[0].legend()  # Adding legend to the first subplot

# Plot the density of 'vo' on the second subplot
ekman_uwCurrent_data_final['vo'].plot(kind='density', title=" ", ax=axes[1],color='green')
axes[1].legend()  # Adding legend to the first subplot

# Adjust layout to avoid overlap
plt.tight_layout()

fig.suptitle("Density of Ekman Data", fontsize=16)


# Show the plot
plt.show()

### QQ plot

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 6))
sm.qqplot(copernicus_uwCurrent_data_final.uo, line='r' ,ax=axes[0])
sm.qqplot(copernicus_uwCurrent_data_final.vo, line='r' ,ax=axes[1])
fig.suptitle("QQplot for Copernicus", fontsize=16)
plt.xlabel("Theoretical Quantiles")
plt.ylabel("Sample Quantiles")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 6))
sm.qqplot(gauss_markov_uwCurrent_data_final['uo'], line='q' ,ax=axes[0])
sm.qqplot(gauss_markov_uwCurrent_data_final['vo'], line='q' ,ax=axes[1])
fig.suptitle("QQplot for Gaussian", fontsize=16)
plt.xlabel("Theoretical Quantiles")
plt.ylabel("Sample Quantiles")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 6))
sm.qqplot(gauss_markov_expanded['uo'], line='q' ,ax=axes[0])
sm.qqplot(gauss_markov_expanded['vo'], line='q' ,ax=axes[1])
fig.suptitle("QQplot for Gaussian Expanded", fontsize=16)
plt.xlabel("Theoretical Quantiles")
plt.ylabel("Sample Quantiles")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 6))
sm.qqplot(ekman_uwCurrent_data_final['uo'], line='r' ,ax=axes[0])
sm.qqplot(ekman_uwCurrent_data_final['vo'], line='r' ,ax=axes[1])
fig.suptitle("QQplot for Ekman", fontsize=16)
plt.xlabel("Theoretical Quantiles")
plt.ylabel("Sample Quantiles")
plt.tight_layout()
plt.show()

# 3. Data visualisation on 2 days

Display coordinates and velocity values of dataframes

In [ ]:
## Display dataframes coordinates
fig, axes = plt.subplots(1, 3, figsize=(15, 5))  # 1 ligne, 3 colonnes

# Plot Bathy dataframe
axes[0].scatter(copernicus_bathy_data_cart['longitude'], copernicus_bathy_data_cart['latitude'], color='blue', marker='o')
axes[0].set_title('Bathy')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].grid(True)

# Plot UW Current dataframe
axes[1].scatter(copernicus_uwCurrent_data_cart['longitude'], copernicus_uwCurrent_data_cart['latitude'], color='green', marker='^')
axes[1].set_title('UWCurrent')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].grid(True)

# Plot Wind dataframe
axes[2].scatter(copernicus_wind_data_cart['longitude'], copernicus_wind_data_cart['latitude'], color='red', marker='s')
axes[2].set_title('Wind')
axes[2].set_xlabel('Longitude')
axes[2].set_ylabel('Latitude')
axes[2].grid(True)

# Adjust spaces between subplots
plt.tight_layout()

# Show graph
plt.show()

In [ ]:
## Display dataframes velocities or depths

# Création d'un graphique
fig = plt.figure(figsize=(12, 6))

# Premier subplot (subgraphique 1)
ax1 = fig.add_subplot(131)
# Tracer les flèches 2D
ax1.quiver(copernicus_wind_data['longitude'], copernicus_wind_data['latitude'],
          copernicus_wind_data['eastward_wind'], copernicus_wind_data['northward_wind'],
          color='blue', linewidth=1, scale = 100)
# Labels et titre
ax1.set_xlabel('longitude')
ax1.set_ylabel('latitude')
ax1.set_title('Wind Velocity')



# Deuxième subplot (subgraphique 2)
ax2 = fig.add_subplot(132, projection='3d')
# Coefficient d'echelle pour les fleches
scaleFactor = 0.5
# Tracer les flèches 3D
ax2.quiver(copernicus_uwCurrent_data['longitude'], copernicus_uwCurrent_data['latitude'], copernicus_uwCurrent_data['depth'],
          copernicus_uwCurrent_data['uo']*scaleFactor, copernicus_uwCurrent_data['vo']*scaleFactor, np.zeros(len(copernicus_uwCurrent_data['vo'])),
          color='blue', linewidth=1)
# Labels et titre
ax2.set_xlabel('longitude')
ax2.set_ylabel('latitude')
ax2.set_zlabel('depth')
ax2.set_title('UW Current Velocity')
ax2.invert_zaxis()  # Inverser l'axe Z

# Troisieme subplot (subgraphique 3)
ax3 = fig.add_subplot(133, projection='3d')
# Tracer les flèches 3D
ax3.plot_trisurf(copernicus_bathy_data['longitude'], copernicus_bathy_data['latitude'], copernicus_bathy_data['deptho'], cmap='viridis', linewidth=0.5)
# Labels et titre
ax3.set_xlabel('longitude')
ax3.set_ylabel('latitude')
ax3.set_zlabel('depth')
ax3.set_title('Bathy')
ax3.invert_zaxis()  # Inverser l'axe Z

# Afficher le graphique
plt.show()

In [ ]:
# Animate display
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation

# On recupere toutes les differentes dates de releve IN SITU
time_array = copernicus_uwCurrent_data.time.unique()
longitude = copernicus_uwCurrent_data[copernicus_uwCurrent_data['time']==time_array[1]]['longitude']
latitude = copernicus_uwCurrent_data[copernicus_uwCurrent_data['time']==time_array[1]]['latitude']
depth = copernicus_uwCurrent_data[copernicus_uwCurrent_data['time']==time_array[1]]['depth']
W = np.zeros(len(longitude))

fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(111, projection='3d')

def get_uwcurrent(temps):
    U = copernicus_uwCurrent_data[copernicus_uwCurrent_data['time']==temps]['uo']
    V = copernicus_uwCurrent_data[copernicus_uwCurrent_data['time']==temps]['vo']
    return U,V

quiver = ax.quiver(longitude, latitude, depth, *get_uwcurrent(time_array[1]), W)

# Labels et titre
ax.set_xlabel('longitude')
ax.set_ylabel('latitude')
ax.set_zlabel('depth')
ax.set_title('Vitesse du courant')
ax.invert_zaxis()  # Inverser l'axe Z

def update(temps):
    global quiver
    quiver.remove()
    quiver = ax.quiver(longitude, latitude, depth, *get_uwcurrent(temps), W)
    print('tt')

ani = FuncAnimation(fig, update, frames=time_array[1:])
plt.show()

# 4. Bivariate statistics

## 4.1. Correlation between Uo and Vo 

### Copernicus

Signifiant correlation between U0 and V0

In [ ]:
copernicus_u_v_corr, copernicus_u_v_corr_p_value = pearsonr(copernicus_uwCurrent_data_final['uo'], copernicus_uwCurrent_data_final['vo'])
print("p-value:",copernicus_u_v_corr_p_value.round(3))
print("coef: ",copernicus_u_v_corr.round(3))

### Ekman 

In [ ]:
ekman_u_v_corr, ekman_u_v_corr_p_value = pearsonr(ekman_uwCurrent_data_final['uo'], ekman_uwCurrent_data_final['vo'])
print("p-value:",ekman_u_v_corr_p_value.round(3))
print("coef: ",ekman_u_v_corr.round(3))

## 4.2 Correlation between Copernicus and Gauss-Markov (UUV Simulator) and Ekman

### Root Mean Square Error (RMSE)

<b>RMSE</b> is a simple and intuitive measure to compare the discrepancies between corresponding components of two DataFrames.<br>
Lower RMSE values indicate that the datasets are similar, whereas higher values indicate greater differences.<br>
This method provides an alternative to R² when you are interested in the magnitude of errors rather than the proportion of variance explained.

In [ ]:
def calculate_rmse(actual, predicted):
    return np.sqrt(((actual - predicted) ** 2).mean())

In [ ]:
# UW Current
## Calculation of the RMSE with Gauss-Markov not expanded for the columns 'u' and 'v'.
current_rmse_u1_gauss_markov = calculate_rmse(copernicus_uwCurrent_data_final['uo'], gauss_markov_uwCurrent_data_final['uo'])
current_rmse_v1_gauss_markov = calculate_rmse(copernicus_uwCurrent_data_final['vo'], gauss_markov_uwCurrent_data_final['vo'])

In [ ]:
# UW Current
## Calculation of the RMSE with Gauss-Markov expanded not expanded for the columns 'u' and 'v'.
current_rmse_u1_gauss_markov_expanded = calculate_rmse(copernicus_uwCurrent_data_final['uo'].reset_index(drop=True), gauss_markov_expanded['uo'].reset_index(drop=True))
current_rmse_v1_gauss_markov_expanded = calculate_rmse(copernicus_uwCurrent_data_final['vo'].reset_index(drop=True), gauss_markov_expanded['vo'].reset_index(drop=True))

In [ ]:
# UW Current
## Calculation of the RMSE with Ekman for the columns 'u' and 'v'.
current_rmse_u1_ekman = calculate_rmse(copernicus_uwCurrent_data_final['uo'], ekman_uwCurrent_data_final['uo'])
current_rmse_v1_ekman = calculate_rmse(copernicus_uwCurrent_data_final['vo'], ekman_uwCurrent_data_final['vo'])

In [ ]:
# Compute mean of actual values (Copernicus data)
mean_uo = np.mean(np.abs(copernicus_uwCurrent_data_final['uo']))
mean_vo = np.mean(np.abs(copernicus_uwCurrent_data_final['vo']))

### Copernicus vs Gauss Markov

In [ ]:
# Compute RMSE as percentage
relative_rmse_u1_gauss_markov = (current_rmse_u1_gauss_markov / mean_uo) * 100
relative_rmse_v1_gauss_markov = (current_rmse_v1_gauss_markov / mean_vo) * 100

print("RMSE between Copernicus and Gauss-Markov\n")
print(f"RMSE u: {current_rmse_u1_gauss_markov} m/s ({relative_rmse_u1_gauss_markov:.2f}%) for the u-component (east-west current)")
print(f"RMSE v: {current_rmse_v1_gauss_markov} m/s ({relative_rmse_v1_gauss_markov:.2f}%) for the v-component (north-south current)")

### Copernicus vs Gauss Markov expanded

In [ ]:
# Compute RMSE as percentage
relative_rmse_u1_gauss_markov_expanded = (current_rmse_u1_gauss_markov_expanded / mean_uo) * 100
relative_rmse_v1_gauss_markov_expanded = (current_rmse_v1_gauss_markov_expanded / mean_vo) * 100

print("RMSE between Copernicus and Gauss-Markov Expanded\n")
print(f"RMSE u: {current_rmse_u1_gauss_markov_expanded} m/s ({relative_rmse_u1_gauss_markov_expanded:.2f}%) for the u-component (east-west current)")
print(f"RMSE v: {current_rmse_v1_gauss_markov_expanded} m/s ({relative_rmse_v1_gauss_markov_expanded:.2f}%) for the v-component (north-south current)")

### Copernicus vs Ekman

In [ ]:
# Compute RMSE as percentage
relative_rmse_u1_ekman = (current_rmse_u1_ekman / mean_uo) * 100
relative_rmse_v1_ekman = (current_rmse_v1_ekman / mean_vo) * 100

print("\n\nRMSE between Copernicus and Ekman\n")
print(f"RMSE u: {current_rmse_u1_ekman} m/s ({relative_rmse_u1_ekman:.2f}%) for the u-component (east-west current)")
print(f"RMSE v: {current_rmse_v1_ekman} m/s ({relative_rmse_v1_ekman:.2f}%) for the v-component (north-south current)")

### Mean Absolute Error (MAE)

<b>MAE</b> is a useful and simple metric to compare how well two datasets align, particularly when you're interested in the magnitude of the error rather than the variance explained (like R²).<br>
Lower MAE values indicate better alignment, while higher values indicate greater discrepancies between the two datasets.

In [ ]:
def calculate_mae(actual, predicted):
    return np.mean(np.abs(actual - predicted))

### Copernicus vs Gauss Markov

In [ ]:
## Calculation of the MAE for the columns 'u' and 'v'.
# UW Current
current_mae_u1 = calculate_mae(copernicus_uwCurrent_data_final['uo'], gauss_markov_uwCurrent_data_final['uo'])
current_mae_v1 = calculate_mae(copernicus_uwCurrent_data_final['vo'], gauss_markov_uwCurrent_data_final['vo'])
print(f"uw current MAE between 'u' in copernicus_uwCurrent_data_cart and gauss_noise_copernicus_uwCurrent_data_cart: {current_mae_u1}")
print(f"uw current MAE between 'v' in copernicus_uwCurrent_data_cart and gauss_noise_copernicus_uwCurrent_data_cart: {current_mae_v1}")

### Copernicus vs Gauss Markov expanded

In [ ]:
## Calculation of the MAE for the columns 'u' and 'v'.
# UW Current
current_mae_u2 = calculate_mae(copernicus_uwCurrent_data_final['uo'], gauss_markov_expanded['uo'].reset_index(drop=True))
current_mae_v2 = calculate_mae(copernicus_uwCurrent_data_final['vo'], gauss_markov_expanded['vo'].reset_index(drop=True))
print(f"uw current MAE between 'u' in copernicus_uwCurrent_data_cart and gauss_markov_expanded: {current_mae_u2}")
print(f"uw current MAE between 'v' in copernicus_uwCurrent_data_cart and gauss_markov_expanded: {current_mae_v2}")

### Copernicus vs Ekman

In [ ]:
current_mae_u3 = calculate_mae(copernicus_uwCurrent_data_final['uo'], ekman_uwCurrent_data_final['uo'])
current_mae_v3 = calculate_mae(copernicus_uwCurrent_data_final['vo'], ekman_uwCurrent_data_final['vo'])
# Print results
print(f"\nuw current MAE between 'u' in copernicus_uwCurrent_data_cart and ekman_uwCurrent_data_cart: {current_mae_u3}")
print(f"uw current MAE between 'v' in copernicus_uwCurrent_data_cart and ekman_uwCurrent_data_cart: {current_mae_v3}")

# 5. Statistics at different depths

In [ ]:
# Define depth categories
depth_ranges = {
    "Surface (0-100m)": (0, 100),
    "Shallow (100-200m)": (100, 200),
    "Mid-depth (200-500m)": (200, 500),
    "Deep (500m+)": (500, np.inf)
}

In [ ]:
def compute_errors(df1, df2, depth_col, u_col, v_col):
    results = []
    for label, (dmin, dmax) in depth_ranges.items():
        subset1 = df1[(df1[depth_col] >= dmin) & (df1[depth_col] < dmax)]
        subset2 = df2[(df2[depth_col] >= dmin) & (df2[depth_col] < dmax)]
        
        if len(subset1) == 0 or len(subset2) == 0:
            continue  # Skip if no data
        
        mae_u = calculate_mae(subset1[u_col], subset2[u_col])
        mae_v = calculate_mae(subset1[v_col], subset2[v_col])
        rmse_u = calculate_rmse(subset1[u_col], subset2[u_col])
        rmse_v = calculate_rmse(subset1[v_col], subset2[v_col])
        
        results.append((label, mae_u, mae_v, rmse_u, rmse_v))
    
    return pd.DataFrame(results, columns=["Depth Range", "MAE u", "MAE v", "RMSE u", "RMSE v"])

In [ ]:
# Compute Gauss_Markov Expanded errors
gauss_errors = compute_errors(copernicus_uwCurrent_data_final, gauss_markov_expanded.reset_index(drop=True), "depth", "uo", "vo")

In [ ]:
# Compute Ekman errors
ekman_errors = compute_errors(copernicus_uwCurrent_data_final, ekman_uwCurrent_data_final, "depth", "uo", "vo")

In [ ]:
# Plot Results
def plot_errors(errors, model_name):
    plt.figure(figsize=(10, 5))
    plt.bar(errors["Depth Range"], errors["RMSE u"], label="RMSE u", alpha=0.7)
    plt.bar(errors["Depth Range"], errors["RMSE v"], label="RMSE v", alpha=0.7)
    plt.title(f"{model_name} RMSE at Different Depths")
    plt.ylabel("RMSE (m/s)")
    plt.legend()
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
plot_errors(gauss_errors, "Gauss-Markov Expanded Model")

In [ ]:
plot_errors(ekman_errors, "Ekman Model")

In [ ]:
print("\nGauss-Markov Expanded Model Errors:\n", gauss_errors)

In [ ]:
print("\nEkman Model Errors:\n", ekman_errors)

In [ ]:
# Calculate percentage improvement
def percentage_improvement(ekman, gauss_markov):
    return (1 - ekman / gauss_markov) * 100

improvement_df = ekman_errors.copy()
for metric in ["MAE u", "MAE v", "RMSE u", "RMSE v"]:
    improvement_df[metric] = percentage_improvement(ekman_errors[metric], gauss_errors[metric])

print("Percentage Improvement Using Ekman Model (compared to Gauss-Markov):")
print(improvement_df)

## Plot  

In [ ]:
# Filter data for specific lat/lon, depth range, and time
copernicus_filtered_data = copernicus_uwCurrent_data_final[
    (copernicus_uwCurrent_data_final['latitude'] == 47) &
    (copernicus_uwCurrent_data_final['longitude'] == -6) &
    (copernicus_uwCurrent_data_final['depth'] >= 0) &
    (copernicus_uwCurrent_data_final['depth'] <= 300) &
    (copernicus_uwCurrent_data_final['time'].apply(lambda x: x.strftime('%H:%M:%S')) == "18:00:00") 
]

# Sort by depth
copernicus_filtered_data = copernicus_filtered_data.sort_values(by='depth')

# Plot
plt.figure(figsize=(8, 6))
plt.plot(copernicus_filtered_data['uo'], copernicus_filtered_data['depth'], marker='o', color='b')
plt.title("Copernicus: u0 vs Depth at (47, -6) at 18:00:00")
plt.xlabel("u0 (m/s)")
plt.ylabel("Depth (m)")
plt.gca().invert_yaxis()  # Invert the depth axis
plt.grid(True)
plt.show()

In [ ]:
# Filter data for specific lat/lon, depth range, and time
gauss_expanded_filtered_data = gauss_markov_expanded[
    (gauss_markov_expanded['latitude'] == 47) &
    (gauss_markov_expanded['longitude'] == -6) &
    (gauss_markov_expanded['depth'] >= 0) &
    (gauss_markov_expanded['depth'] <= 500) & (gauss_markov_expanded['time'] == "18:00:00")
]

# Sort by depth
gauss_expanded_filtered_data = gauss_expanded_filtered_data.sort_values(by='depth')

# Plot
plt.figure(figsize=(8, 6))
plt.plot(gauss_expanded_filtered_data['uo'], gauss_expanded_filtered_data['depth'], marker='o', color='b')
plt.title("Gauss Markov expanded: u0 vs Depth at (47, -6) at 18:00:00")
plt.xlabel("u0 (m/s)")
plt.ylabel("Depth (m)")
plt.gca().invert_yaxis()  # Invert the depth axis
plt.grid(True)
plt.show()

In [ ]:
# Filter data for specific lat/lon, depth range, and time
ekman_filtered_data = ekman_uwCurrent_data_final[
    (ekman_uwCurrent_data_final['latitude'] == 47) &
    (ekman_uwCurrent_data_final['longitude'] == -6) &
    (ekman_uwCurrent_data_final['depth'] >= 0) &
    (ekman_uwCurrent_data_final['depth'] <= 50) & (ekman_uwCurrent_data_final['time'] == "18:00:00")
]

# Sort by depth
ekman_filtered_data = ekman_filtered_data.sort_values(by='depth')

# Plot 
plt.figure(figsize=(8, 6))
plt.plot(ekman_filtered_data['uo'], ekman_filtered_data['depth'], marker='o', color='b')
plt.title("Ekman: u0 vs Depth at (47, -6) at 18:00:00")
plt.xlabel("u0 (m/s)")
plt.ylabel("Depth (m)")
plt.gca().invert_yaxis()  # Invert the depth axis
plt.grid(True)
plt.show()

In [ ]:
# Ensure 'time' column is properly formatted
for model in [ekman_uwCurrent_data_final, gauss_markov_expanded.reset_index(drop=True), copernicus_uwCurrent_data_final]:
    model['time'] = pd.to_datetime(model['time'], format='%H:%M:%S', errors='coerce')  # Converts everything safely

# Define models and their properties
models = {
    "Ekman": {
        "data": ekman_uwCurrent_data_final,
        "depth_range": (0, 300),
        "color": "b"
    },
    "Gauss Markov": {
        "data": gauss_markov_expanded,
        "depth_range": (0, 300),
        "color": "r"
    },
    "Copernicus": {
        "data": copernicus_uwCurrent_data_final,
        "depth_range": (0, 300),
        "color": "g"
    }
}

In [ ]:
# Create figure
plt.figure(figsize=(10, 8))

# Loop over models to filter data and plot
for model_name, properties in models.items():
    df = properties["data"]
    min_depth, max_depth = properties["depth_range"]

    # Convert 'time' column to datetime if it's not already
    df['time'] = pd.to_datetime(df['time'])
    # Filter data
    filtered_data = df[
        (df['latitude'] == 47) &
        (df['longitude'] == -6) &
        (df['depth'] >= min_depth) &
        (df['depth'] <= max_depth) &
        (df['time'].dt.strftime('%H:%M:%S') == "18:00:00")  # Use .dt to access datetime attributes
    ].sort_values(by='depth')

    # Plot with unique color for each model
    plt.plot(filtered_data['uo'], filtered_data['depth'], marker='o', linestyle='-', label=model_name, color=properties["color"])

# Configure plot
plt.title("u0 vs Depth at (47, -6) at 18:00:00")
plt.xlabel("u0 (m/s)")
plt.ylabel("Depth (m)")
plt.gca().invert_yaxis()  # Invert the depth axis (0m at top, increasing downward)
plt.grid(True)
plt.legend()  # Show legend for models

# Show the plot
plt.show()

In [ ]:
# Loop over models to filter data, compute mean u0, and plot
for model_name, properties in models.items():
    df = properties["data"]
    min_depth, max_depth = properties["depth_range"]
    
    # Filter data for time and depth range
    filtered_data = df[
        (df['depth'] >= min_depth) &
        (df['depth'] <= max_depth) &
        (df['time'].dt.strftime('%H:%M:%S') == "18:00:00")  # Use .dt to access datetime attributes
    ]
    
    # Compute mean uo for each depth
    mean_uo = filtered_data.groupby('depth')['uo'].mean().reset_index()

    # Plot with unique color for each model
    plt.plot(mean_uo['uo'], mean_uo['depth'], marker='o', linestyle='-', label=model_name, color=properties["color"])

# Configure plot
plt.title("Mean u0 vs Depth at 18:00:00 (Averaged Over All Lat/Lon)")
plt.xlabel("Mean u0 (m/s)")
plt.ylabel("Depth (m)")
plt.gca().invert_yaxis()  # Invert the depth axis (0m at top, increasing downward)
plt.grid(True)
plt.legend()  # Show legend for models

# Show the plot
plt.show()

In [ ]:
# Loop over models to filter data, compute mean u0 and variance, and plot
for model_name, properties in models.items():
    df = properties["data"]
    min_depth, max_depth = properties["depth_range"]
    
    # Filter data for time and depth range
    filtered_data = df[
        (df['depth'] >= min_depth) &
        (df['depth'] <= max_depth) &
        (df['time'].dt.strftime('%H:%M:%S') == "18:00:00")
    ]
    
    # Compute mean and variance of uo for each depth
    stats_uo = filtered_data.groupby('depth')['uo'].agg(['mean', 'var']).reset_index()

    # Extract mean and variance
    mean_uo = stats_uo['mean']
    var_uo = stats_uo['var'].fillna(0)  # Fill NaN values (in case of single data point per depth)

    # Compute standard deviation as error bars
    std_uo = var_uo ** 0.5

    # Plot mean with shaded variance
    plt.plot(stats_uo['mean'], stats_uo['depth'], marker='o', linestyle='-', label=model_name, color=properties["color"])
    plt.fill_betweenx(stats_uo['depth'], mean_uo - std_uo, mean_uo + std_uo, color=properties["color"], alpha=0.2)  # Shaded region for variance

# Configure plot
plt.title("Mean u0 vs Depth at 18:00:00 (with Variance)")
plt.xlabel("Mean u0 (m/s)")
plt.ylabel("Depth (m)")
plt.gca().invert_yaxis()  # Invert the depth axis (0m at top, increasing downward)
plt.grid(True)
plt.legend()  # Show legend for models

# Show the plot
plt.show()

In [ ]:
# Loop over models to filter data, compute mean u0, and plot
for model_name, properties in models.items():
    df = properties["data"]
    min_depth, max_depth = properties["depth_range"]
    
    # Filter data for time and depth range
    filtered_data = df[
        (df['depth'] >= min_depth) &
        (df['depth'] <= max_depth) &
        (df['time'].dt.strftime('%H:%M:%S') == "12:00:00")  # Use .dt to access datetime attributes
    ]
    
    # Compute mean uo for each depth
    mean_uo = filtered_data.groupby('depth')['uo'].mean().reset_index()

    # Plot with unique color for each model
    plt.plot(mean_uo['uo'], mean_uo['depth'], marker='o', linestyle='-', label=model_name, color=properties["color"])

# Configure plot
plt.title("Mean u0 vs Depth at 12:00:00 (Averaged Over All Lat/Lon)")
plt.xlabel("Mean u0 (m/s)")
plt.ylabel("Depth (m)")
plt.gca().invert_yaxis()  # Invert the depth axis (0m at top, increasing downward)
plt.grid(True)
plt.legend()  # Show legend for models

# Show the plot
plt.show()

In [ ]:
# Loop over models to filter data, compute mean u0 and variance, and plot
for model_name, properties in models.items():
    df = properties["data"]
    min_depth, max_depth = properties["depth_range"]
    
    # Filter data for time and depth range
    filtered_data = df[
        (df['depth'] >= min_depth) &
        (df['depth'] <= max_depth) &
        (df['time'].dt.strftime('%H:%M:%S') == "12:00:00")
    ]
    
    # Compute mean and variance of uo for each depth
    stats_uo = filtered_data.groupby('depth')['uo'].agg(['mean', 'var']).reset_index()

    # Extract mean and variance
    mean_uo = stats_uo['mean']
    var_uo = stats_uo['var'].fillna(0)  # Fill NaN values (in case of single data point per depth)

    # Compute standard deviation as error bars
    std_uo = var_uo ** 0.5

    # Plot mean with shaded variance
    plt.plot(stats_uo['mean'], stats_uo['depth'], marker='o', linestyle='-', label=model_name, color=properties["color"])
    plt.fill_betweenx(stats_uo['depth'], mean_uo - std_uo, mean_uo + std_uo, color=properties["color"], alpha=0.2)  # Shaded region for variance

# Configure plot
plt.title("Mean u0 vs Depth at 12:00:00 (with Variance)")
plt.xlabel("Mean u0 (m/s)")
plt.ylabel("Depth (m)")
plt.gca().invert_yaxis()  # Invert the depth axis (0m at top, increasing downward)
plt.grid(True)
plt.legend()  # Show legend for models

# Show the plot
plt.show()

In [ ]:
# Loop over models to filter data, compute mean u0, and plot
for model_name, properties in models.items():
    df = properties["data"]
    min_depth, max_depth = properties["depth_range"]
    
    # Filter data for time and depth range
    filtered_data = df[
        (df['depth'] >= min_depth) &
        (df['depth'] <= max_depth) &
        (df['time'].dt.strftime('%H:%M:%S') == "06:00:00")  # Use .dt to access datetime attributes
    ]
    
    # Compute mean uo for each depth
    mean_uo = filtered_data.groupby('depth')['uo'].mean().reset_index()

    # Plot with unique color for each model
    plt.plot(mean_uo['uo'], mean_uo['depth'], marker='o', linestyle='-', label=model_name, color=properties["color"])

# Configure plot
plt.title("Mean u0 vs Depth at 06:00:00 (Averaged Over All Lat/Lon)")
plt.xlabel("Mean u0 (m/s)")
plt.ylabel("Depth (m)")
plt.gca().invert_yaxis()  # Invert the depth axis (0m at top, increasing downward)
plt.grid(True)
plt.legend()  # Show legend for models

# Show the plot
plt.show()

In [ ]:
# Loop over models to filter data, compute mean u0 and variance, and plot
for model_name, properties in models.items():
    df = properties["data"]
    min_depth, max_depth = properties["depth_range"]
    
    # Filter data for time and depth range
    filtered_data = df[
        (df['depth'] >= min_depth) &
        (df['depth'] <= max_depth) &
        (df['time'].dt.strftime('%H:%M:%S') == "06:00:00")
    ]
    
    # Compute mean and variance of uo for each depth
    stats_uo = filtered_data.groupby('depth')['uo'].agg(['mean', 'var']).reset_index()

    # Extract mean and variance
    mean_uo = stats_uo['mean']
    var_uo = stats_uo['var'].fillna(0)  # Fill NaN values (in case of single data point per depth)

    # Compute standard deviation as error bars
    std_uo = var_uo ** 0.5

    # Plot mean with shaded variance
    plt.plot(stats_uo['mean'], stats_uo['depth'], marker='o', linestyle='-', label=model_name, color=properties["color"])
    plt.fill_betweenx(stats_uo['depth'], mean_uo - std_uo, mean_uo + std_uo, color=properties["color"], alpha=0.2)  # Shaded region for variance

# Configure plot
plt.title("Mean u0 vs Depth at 06:00:00 (with Variance)")
plt.xlabel("Mean u0 (m/s)")
plt.ylabel("Depth (m)")
plt.gca().invert_yaxis()  # Invert the depth axis (0m at top, increasing downward)
plt.grid(True)
plt.legend()  # Show legend for models

# Show the plot
plt.show()

In [ ]:
# Loop over models to filter data, compute mean u0, and plot
for model_name, properties in models.items():
    df = properties["data"]
    min_depth, max_depth = properties["depth_range"]
    
    # Filter data for time and depth range
    filtered_data = df[
        (df['depth'] >= min_depth) &
        (df['depth'] <= max_depth) &
        (df['time'].dt.strftime('%H:%M:%S') == "00:00:00")  # Use .dt to access datetime attributes
    ]
    
    # Compute mean uo for each depth
    mean_uo = filtered_data.groupby('depth')['uo'].mean().reset_index()

    # Plot with unique color for each model
    plt.plot(mean_uo['uo'], mean_uo['depth'], marker='o', linestyle='-', label=model_name, color=properties["color"])

# Configure plot
plt.title("Mean u0 vs Depth at 00:00:00 (Averaged Over All Lat/Lon)")
plt.xlabel("Mean u0 (m/s)")
plt.ylabel("Depth (m)")
plt.gca().invert_yaxis()  # Invert the depth axis (0m at top, increasing downward)
plt.grid(True)
plt.legend()  # Show legend for models

# Show the plot
plt.show()

In [ ]:
# Loop over models to filter data, compute mean u0 and variance, and plot
for model_name, properties in models.items():
    df = properties["data"]
    min_depth, max_depth = properties["depth_range"]
    
    # Filter data for time and depth range
    filtered_data = df[
        (df['depth'] >= min_depth) &
        (df['depth'] <= max_depth) &
        (df['time'].dt.strftime('%H:%M:%S') == "00:00:00")
    ]
    
    # Compute mean and variance of uo for each depth
    stats_uo = filtered_data.groupby('depth')['uo'].agg(['mean', 'var']).reset_index()

    # Extract mean and variance
    mean_uo = stats_uo['mean']
    var_uo = stats_uo['var'].fillna(0)  # Fill NaN values (in case of single data point per depth)

    # Compute standard deviation as error bars
    std_uo = var_uo ** 0.5

    # Plot mean with shaded variance
    plt.plot(stats_uo['mean'], stats_uo['depth'], marker='o', linestyle='-', label=model_name, color=properties["color"])
    plt.fill_betweenx(stats_uo['depth'], mean_uo - std_uo, mean_uo + std_uo, color=properties["color"], alpha=0.2)  # Shaded region for variance

# Configure plot
plt.title("Mean u0 vs Depth at 00:00:00 (with Variance)")
plt.xlabel("Mean u0 (m/s)")
plt.ylabel("Depth (m)")
plt.gca().invert_yaxis()  # Invert the depth axis (0m at top, increasing downward)
plt.grid(True)
plt.legend()  # Show legend for models

# Show the plot
plt.show()

# 6 Global MAE/RMSE Metrics 

In [ ]:
# Define depth ranges
def depth_range(depth):
    if depth < 100:
        return 'Surface (0-100m)'
    elif depth < 200:
        return 'Shallow (100-200m)'
    elif depth < 500:
        return 'Mid-depth (200-500m)'
    else:
        return 'Deep (500m+)'

In [ ]:
ekman = ekman_uwCurrent_data_final.copy()
copernicus = copernicus_uwCurrent_data_final.copy()
gauss = gauss_markov_expanded.copy()

In [ ]:
# Round depth to merge (optional, depending on your matching criteria)
for df in [ekman, gauss, copernicus]:
    df['Depth_Range'] = df['depth'].apply(depth_range)
    df['depth_round'] = df['depth'].round(2)  
    df['time'] = pd.to_datetime(df['time'], format='%H:%M:%S', errors='coerce')
    df['lat_round'] = df['latitude'].round(4)
    df['lon_round'] = df['longitude'].round(4)

In [ ]:
# Merge Ekman with Copernicus
ekman_merge = pd.merge(
    ekman,
    copernicus,
    left_on=['depth_round', 'lat_round', 'lon_round', 'time'],
    right_on=['depth_round', 'lat_round', 'lon_round', 'time'],
    suffixes=('_ekman', '_copernicus')
)

In [ ]:
# Merge Gauss-Markov with Copernicus
gauss_merge = pd.merge(
    gauss,
    copernicus,
    left_on=['depth_round', 'lat_round', 'lon_round', 'time'],
    right_on=['depth_round', 'lat_round', 'lon_round', 'time'],
    suffixes=('_gauss', '_copernicus')
)

In [ ]:
# After merging, keep Copernicus Depth_Range
ekman_merge['Depth_Range'] = ekman_merge['Depth_Range_copernicus']
gauss_merge['Depth_Range'] = gauss_merge['Depth_Range_copernicus']

In [ ]:
# Compute vector errors relative to Copernicus
ekman_merge['vector_error'] = np.sqrt(
    (ekman_merge['uo_ekman'] - ekman_merge['uo_copernicus'])**2 +
    (ekman_merge['vo_ekman'] - ekman_merge['vo_copernicus'])**2
)

In [ ]:
gauss_merge['vector_error'] = np.sqrt(
    (gauss_merge['uo_gauss'] - gauss_merge['uo_copernicus'])**2 +
    (gauss_merge['vo_gauss'] - gauss_merge['vo_copernicus'])**2
)

In [ ]:
# Group by Depth Range to get MAE
mae_ekman = ekman_merge.groupby('Depth_Range')['vector_error'].mean().reset_index()
mae_ekman.rename(columns={'vector_error': 'MAE_Ekman'}, inplace=True)

mae_gauss = gauss_merge.groupby('Depth_Range')['vector_error'].mean().reset_index()
mae_gauss.rename(columns={'vector_error': 'MAE_Gauss'}, inplace=True)

In [ ]:
rmse_ekman = ekman_merge.groupby('Depth_Range').apply(
    lambda x: np.sqrt((x['vector_error']**2).mean())
).reset_index(name='RMSE_Ekman')

rmse_gauss = gauss_merge.groupby('Depth_Range').apply(
    lambda x: np.sqrt((x['vector_error']**2).mean())
).reset_index(name='RMSE_Gauss')

In [ ]:
# Define the desired depth order (surface first, deep last)
depth_order = ['Surface (0-100m)', 'Shallow (100-200m)', 'Mid-depth (200-500m)', 'Deep (500m+)']

In [ ]:
# Merge results to compare
mae_comparison = pd.merge(mae_ekman, mae_gauss, on='Depth_Range')

In [ ]:
# Assuming mae_comparison is your DataFrame
mae_comparison['Improvement_MAE_%'] = (
    (mae_comparison['MAE_Gauss'] - mae_comparison['MAE_Ekman']) / mae_comparison['MAE_Gauss']
) * 100

In [ ]:
# Merge results to compare
rmse_comparison = pd.merge(rmse_ekman, rmse_gauss, on='Depth_Range')

In [ ]:
# Assuming rmse_comparison is your DataFrame
rmse_comparison['Improvement_RMSE_%'] = (
    (rmse_comparison['RMSE_Gauss'] - rmse_comparison['RMSE_Ekman']) / rmse_comparison['RMSE_Gauss']
) * 100

In [ ]:
# Define the desired depth order (surface first, deep last)
depth_order = ['Surface (0-100m)', 'Shallow (100-200m)', 'Mid-depth (200-500m)', 'Deep (500m+)']

# Convert Depth_Range to categorical with this order
mae_comparison['Depth_Range'] = pd.Categorical(mae_comparison['Depth_Range'],
                                               categories=depth_order,
                                               ordered=True)

# Sort by this categorical order
mae_comparison = mae_comparison.sort_values('Depth_Range').reset_index(drop=True)

rmse_comparison['Depth_Range'] = pd.Categorical(rmse_comparison['Depth_Range'],
                                               categories=depth_order,
                                               ordered=True)

# Sort by this categorical order
rmse_comparison = rmse_comparison.sort_values('Depth_Range').reset_index(drop=True)

In [ ]:
mae_comparison

In [ ]:
rmse_comparison

In [ ]:
# Magnitude
ekman_merge['copernicus_vector'] = np.sqrt(
    ekman_merge['uo_copernicus']**2 + ekman_merge['vo_copernicus']**2
)

In [ ]:
baseline_mean = ekman_merge.groupby('Depth_Range', as_index=False)['copernicus_vector'].mean()
baseline_mean.rename(columns={'copernicus_vector': 'Mean_Copernicus_Vector'}, inplace=True)

In [ ]:
# Ensure Depth_Range matches exactly
mae_comparison['Depth_Range'] = mae_comparison['Depth_Range'].str.strip()
rmse_comparison['Depth_Range'] = rmse_comparison['Depth_Range'].str.strip()
baseline_mean['Depth_Range'] = baseline_mean['Depth_Range'].str.strip()

In [ ]:
# Start building the global table
global_table = pd.merge(mae_comparison, baseline_mean, on='Depth_Range', how='left')

# Merge RMSE columns into global_table
global_table = pd.merge(global_table, 
                        rmse_comparison[['Depth_Range', 'RMSE_Ekman', 'RMSE_Gauss', 'Improvement_RMSE_%']], 
                        on='Depth_Range', 
                        how='left')

In [ ]:
global_table['Rel_MAE_Ekman'] = global_table['MAE_Ekman'] / global_table['Mean_Copernicus_Vector']
global_table['Rel_MAE_Gauss'] = global_table['MAE_Gauss'] / global_table['Mean_Copernicus_Vector']
global_table['Rel_RMSE_Ekman'] = global_table['RMSE_Ekman'] / global_table['Mean_Copernicus_Vector']
global_table['Rel_RMSE_Gauss'] = global_table['RMSE_Gauss'] / global_table['Mean_Copernicus_Vector']

In [ ]:
# Create final summary table
summary_table = global_table[['Depth_Range',
                              'Improvement_MAE_%',
                              'Improvement_RMSE_%',
                              'Rel_MAE_Ekman',
                              'Rel_MAE_Gauss',
                              'Rel_RMSE_Ekman',
                              'Rel_RMSE_Gauss']]

# Round numeric columns to 2 decimals
summary_table = summary_table.round({
    'Improvement_MAE_%': 2,
    'Improvement_RMSE_%': 2,
    'Rel_MAE_Ekman': 2,
    'Rel_MAE_Gauss': 2,
    'Rel_RMSE_Ekman': 2,
    'Rel_RMSE_Gauss': 2
})

summary_table